In [ ]:
# Our first set of imports will all be our backbone library for all of our models, Torch

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

# While torch takes care of most of the data-loading, torch vision will be what we need for the computer "vision" part of this tutorial
import torchvision
from torchvision.transforms import v2
from torchvision.models import resnet50, ResNet50_Weights
import torchvision.models as models
#from torchmetrics.classification import MulticlassAccuracy
from transformers import AutoProcessor, AutoModelForImageTextToText
#from lighthouse.models import CGDETRPredictor

#This will be our helper functions for video reading and other general "math" functions
import cv2
from PIL import Image
import numpy as np
import pandas as pd
import math
import random
import os
#import h5py

# Finally, our plotting library
import seaborn as sns
import matplotlib.pyplot as plt

DEVICE = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")

In [2]:
# load and process the manual annotations for the videos
annotations_path = "annotations"
test_annotations_path = os.path.join(annotations_path, "Video_10_merged.txt")
test_annotations = pd.read_csv(test_annotations_path, sep=",")

# Processing the annotations, not sure about the final input format
# Suitable for change in future
test_annotations['average_score'] = 0.0 
for idx, row in test_annotations.iterrows():
    scores = row['combined_salience']
    if scores is not np.nan:
        scores = scores[2:-1].split(";")
        scores = [float(score) for score in scores]
        test_annotations.at[idx, 'average_score'] = np.mean(scores) # Save the average of our salience scoring
    else :
        test_annotations.at[idx, 'average_score'] = np.nan

test_annotations.head()

,Marius,Ionas,Angelos,combined_salience,description,average_score
0,1,Na,Na,(2),News start,2.000000
1,1,1,1,(1; 1; 1),Reporter introduce subject,1.000000
2,1,1,1,(0;1;1),People talking around EV,0.666667
3,1,Na,Na,(2),Women interview man,2.000000
4,1,1,1,(2; 2; 2),EV being plugged it to charge,2.000000


In [3]:
video_folder_path = "data"
videos_path = os.listdir(video_folder_path)
videos_path = [os.path.join(video_folder_path, video) for video in videos_path]
print(videos_path)

['data/video_11.mp4', 'data/video_12.mp4', 'data/video_10.mp4', 'data/video_9.mp4']


## Event detection using Vision Language models (VLMs)

In [4]:
# Downloading the model
model_path = "HuggingFaceTB/SmolVLM2-2.2B-Instruct"
processor = AutoProcessor.from_pretrained(model_path)

model = AutoModelForImageTextToText.from_pretrained(
    model_path,
    dtype=torch.bfloat16,
).to(DEVICE)

Loading weights:   0%|          | 0/657 [00:00<?, ?it/s]

In [6]:
messages = [
    {
        "role": "user",
        "content": [
            {"type": "video", "path": videos_path[2]}, # Select video 10 
            {"type": "text", "text": "Describe the most relevant events in the video, list each event sequentially, using a numbered format. Describe it in terms of the actions different persons take"}
        ]
    },
]

inputs = processor.apply_chat_template(
    messages,
    num_frames = 15,
    add_generation_prompt=True,
    tokenize=True,
    return_dict=True,
    return_tensors="pt",
).to(model.device, dtype=torch.bfloat16) # Considering the size of the model, and the number of frames we are sampling are few. This is ultimately a choice that you must make

generated_ids = model.generate(**inputs, do_sample=False, max_new_tokens=128)
events_detected = processor.batch_decode(
    generated_ids,
    skip_special_tokens=True,
)

print(events_detected[0])

/home/ionas/miniforge3/envs/CV_project/lib/python3.12/site-packages/transformers/video_processing_utils.py:872: UserWarning: `torchcodec` is not installed and cannot be used to decode the video by default. Falling back to `torchvision`. Note that `torchvision` decoding is deprecated and will be removed in future versions. 
  warnings.warn(
/home/ionas/miniforge3/envs/CV_project/lib/python3.12/site-packages/transformers/video_utils.py:534: UserWarning: Using `torchvision` for video decoding is deprecated and will be removed in future versions. Please use `torchcodec` instead.
  warnings.warn(


AttributeError: module 'torchvision.io' has no attribute 'read_video'

In [1]:
import torchvision.io
# Check if this works instead
reader = torchvision.io.VideoReader("video.mp4", "video")

AttributeError: module 'torchvision.io' has no attribute 'VideoReader'

## Moment-Retrivial

In [1]:
from lighthouse.models import CGDETRPredictor

model = CGDETRPredictor('results/cg_detr/qvhighlight/clip/best.ckpt', device=DEVICE, feature_name='clip')

# Extract the moments for each event detected by the VLM 
video = model.encode_video(videos_path[2]) # Select video 10
predictions = [model.predict(query, video) for query in events_detected]

print("Moments:")
for i in range(len(predictions)):
    print(f"- Query: {events_detected[i]}")
    print(f"- Predicted moment: {predictions[i]}\n")

ModuleNotFoundError: No module named 'lighthouse'